## CHMM Distillation - Uniform-8 Clone Schedule

**Clone schedule used here**

- Every token receives exactly 8 clones.
- This is a simple uniform-capacity baseline against the frequency-aware schedules.

In [1]:
import os
from pathlib import Path
from transformers import AutoTokenizer

BASE_MODEL_PATH = 'gpt2-large'
TOKENIZER_NAME_OR_PATH = BASE_MODEL_PATH
INPUT_FILE = './workspace/inference_data/distillation_prompts.json'

DATASET = 'gpt2-large'
DATA_PATH = f'./workspace/hmm_data/{DATASET}'
CUDA_CORES = '0'

LVD_CHUNK_SIZE = 100000
TRAIN_CHUNK_SIZE = 100000
TRAIN_TOTAL_CHUNKS = 100
DEV_CHUNK_SIZE = 10000
MAX_NEW_TOKENS = 32
BATCH_SIZE = 12288

Path(DATA_PATH).mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME_OR_PATH)
VOCAB_SIZE = tokenizer.vocab_size
EOS_TOKEN_ID = tokenizer.eos_token_id

print({'data_path': DATA_PATH, 'vocab_size': VOCAB_SIZE, 'eos_token_id': EOS_TOKEN_ID})


{'data_path': './workspace/hmm_data/gpt2-large', 'vocab_size': 50257, 'eos_token_id': 50256}


In [2]:
import torch

def build_clone_schedule(
    token_frequency: torch.Tensor,
    clone_count: int = 8,
    **_: object,
) -> torch.Tensor:
    vocab_size = token_frequency.shape[0]
    return torch.full((vocab_size,), int(clone_count), dtype=torch.long)


CLONE_SCHEDULE_FUNCTION = './tutorial_distillation_chmm_uniform8.ipynb:build_clone_schedule'


In [3]:
import shlex
import subprocess

MODEL_ID = f'chmm_{DATASET}_uniform8'
MODEL_PATH = f'./workspace/models/{MODEL_ID}'
LOG_FILE = f'./workspace/logs/{MODEL_ID}.log'
TRAINING_LOG_FILE = f'./workspace/logs/{MODEL_ID}_training.log'
PID_FILE = f'./workspace/logs/{MODEL_ID}.pid'
EM_SCHEDULE = '10,1;5,2;4,5;4,10;4,20;1,40'

os.makedirs('./workspace/logs', exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)

cmd = f'''torchrun --standalone --nproc_per_node={len(CUDA_CORES.split(','))} train_chmm.py \
    --model_path {MODEL_PATH} \
    --checkpoint 0 --save_per_step 10 \
    --data_path {DATA_PATH} --dataset {DATASET} --total_chunks {TRAIN_TOTAL_CHUNKS} \
    --batch_size {BATCH_SIZE} --sample_length {MAX_NEW_TOKENS} \
    --em_schedule "{EM_SCHEDULE}" \
    --vocab_size {VOCAB_SIZE} --eos_token_id {EOS_TOKEN_ID} \
    --clone_schedule_file {DATA_PATH}/{DATASET}.lvd \
    --clone_schedule_function {CLONE_SCHEDULE_FUNCTION} \
    --pair_code_chunk_count 0 \
    --dropout 0.0 --pseudocount 0.0003 \
    --log_file {LOG_FILE}'''.strip()

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = CUDA_CORES
env["PYTHONUNBUFFERED"] = "1"

print("Launching:")
print(cmd)
print(f"\nProgress log file: {LOG_FILE}")
print(f"Training log file: {TRAINING_LOG_FILE}")
print(f"PID file: {PID_FILE}")

log_f = open(TRAINING_LOG_FILE, "a", buffering=1)

proc = subprocess.Popen(
    shlex.split(cmd),
    stdout=log_f,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
    env=env,
    start_new_session=True,
    cwd=os.getcwd(),
)

Path(PID_FILE).write_text(str(proc.pid))
log_f.close()
print(f"Started background job with PID {proc.pid}")

Launching:
torchrun --standalone --nproc_per_node=1 train_chmm.py     --model_path ./workspace/models/chmm_gpt2-large_uniform8     --checkpoint 0 --save_per_step 10     --data_path ./workspace/hmm_data/gpt2-large --dataset gpt2-large --total_chunks 100     --batch_size 12288 --sample_length 32     --em_schedule "10,1;5,2;4,5;4,10;4,20;1,40"     --vocab_size 50257 --eos_token_id 50256     --clone_schedule_file ./workspace/hmm_data/gpt2-large/gpt2-large.lvd     --clone_schedule_function ./tutorial_distillation_chmm_uniform8.ipynb:build_clone_schedule     --pair_code_chunk_count 0     --dropout 0.0 --pseudocount 0.0003     --log_file ./workspace/logs/chmm_gpt2-large_uniform8.log

Progress log file: ./workspace/logs/chmm_gpt2-large_uniform8.log
Training log file: ./workspace/logs/chmm_gpt2-large_uniform8_training.log
PID file: ./workspace/logs/chmm_gpt2-large_uniform8.pid
Started background job with PID 318822


In [4]:
!nvidia-smi

Wed Apr 22 16:01:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000000:98:00.0 Off |                   On |
| N/A   61C    P0            162W /  300W |   14739MiB /  81920MiB |     N/A      Default |
|                                         |                        |              Enabled |
+-----------------------------------------+-----